# Matched-control experiment — one victim cell per Kaggle account

1. **Settings:** Accelerator = **GPU T4 x2**, Internet = **On**. Add the input **NIH Chest X-rays** (`nih-chest-xrays/data`).
2. Set `CELL` below (different on each account), then click **Save Version → Save & Run All (Commit)**.
3. If the status says *NOT FINISHED*, add **this notebook's latest output** as an input and Save Version again. It resumes where it stopped.

Full instructions: `Matched_Experiments/kaggle/README_KAGGLE.md` in the repository.

In [ ]:
# ===================== EDIT (different CELL on each account) =====================
CELL  = "std_reg"     # std_reg | fgsm_reg | std_noreg | fgsm_noreg
SEEDS = "42"          # more seeds later, e.g. "42 43 44"
SMOKE = False         # True = ~25-minute pipeline test (writes results_smoke/)
# ==================================================================================
REPO   = "https://github.com/NCRL-IIITB/Evasion_MIA.git"
BRANCH = "main"
CODE   = "/tmp/Evasion_MIA"     # code lives outside /kaggle/working (not saved as output)

In [ ]:
# Get the code (git clone, or git pull if it is already there)
import os, subprocess

url = REPO
try:   # private repository: add a Kaggle secret called GITHUB_TOKEN (Add-ons → Secrets)
    from kaggle_secrets import UserSecretsClient
    url = REPO.replace("https://", "https://" + UserSecretsClient().get_secret("GITHUB_TOKEN") + "@")
except Exception:
    pass

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:   # never print the URL: it may contain the token
        raise RuntimeError("git " + args[0] + " failed:\n" + r.stderr.replace(url, REPO))
    return r.stdout

if os.path.isdir(os.path.join(CODE, ".git")):
    git("-C", CODE, "pull", "--ff-only")
else:
    git("clone", "--depth", "1", "--branch", BRANCH, url, CODE)
print("Code version:", git("-C", CODE, "log", "-1", "--format=%h  %s  (%ci)"))

In [ ]:
# Run the pipeline: restores the previous output (if attached), trains, attacks.
# It stops by itself before Kaggle's 12-hour limit so the output is always saved.
cmd = ["python", "-u", f"{CODE}/Matched_Experiments/kaggle/kaggle_pipeline.py",
       "--cell", CELL, "--seeds", *SEEDS.split()] + (["--smoke"] if SMOKE else [])
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("pipeline exit code:", proc.returncode)

In [ ]:
# Status of this account's cell (also saved as results/STATUS.md in the output)
res = "/kaggle/working/" + ("results_smoke" if SMOKE else "results")
print(open(f"{res}/STATUS.md").read() if os.path.exists(f"{res}/STATUS.md") else "no STATUS.md yet")